# HealthLens: TabPFN-3.5 vs XGBoost vs Logistic Regression
Run every cell top to bottom (Runtime > Run all). A GPU runtime is recommended but not required.

**Before you start:** in the left sidebar open the key icon (Secrets), make sure `TABPFN_TOKEN` exists and *Notebook access* is switched on.
The token is read from there only. It is never printed or written to any file.

What this notebook does:
1. Compares the three models on the same 614/154 split, using the same protocol as before.
2. Times prediction, exact SHAP and the counterfactual search on TabPFN (so you know whether the live app will feel fast enough).
3. Exports `artifacts/model_card.json` and `artifacts/train.csv` for the backend, as a zip.

In [ ]:
!pip -q install tabpfn==10.0.0 xgboost
import tabpfn, importlib.metadata as md_
print('tabpfn', md_.version('tabpfn'))

In [ ]:
import os
from google.colab import userdata
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")   # stays in memory only
print("token loaded:", bool(os.environ["TABPFN_TOKEN"]))

In [ ]:
%%writefile explain_agnostic.py
"""
Model-agnostic explanations for HealthLens (works with TabPFN, XGBoost, LogReg ...).

Why this file exists
--------------------
LinearExplainer is only exact for linear models. With TabPFN we need something that
only needs a `predict_proba` function. With 8 features there are just 2^8 = 256
feature subsets, so we can compute *exact* (interventional) Shapley values by
brute force instead of using a sampling approximation:

    value(S) = average prediction when the features in S take the patient's values
               and the other features are taken from a small background sample.

Additivity always holds:  sum(shap values) == prediction(patient) - average background prediction.
The cost is one batched predict call of 256 x len(background) rows per patient.

Counterfactuals
---------------
`find_counterfactuals` is a simple, deterministic grid search over Glucose and BMI
(the only two features a person can plausibly change). It is NOT the DiCE library.
It needs ONE batched predict call, which is what makes it usable with a slow model.
"""
from __future__ import annotations

from itertools import combinations
from math import factorial
from typing import Callable, Dict, List, Optional, Sequence

import numpy as np

PredictFn = Callable[[np.ndarray], np.ndarray]  # (n, d) float array -> (n,) P(diabetes)


# --------------------------------------------------------------------------------------
# Shapley values
# --------------------------------------------------------------------------------------
class ShapleyExplainer:
    def __init__(
        self,
        predict_fn: PredictFn,
        background: np.ndarray,
        feature_names: Sequence[str],
        chunk_rows: int = 4096,
    ):
        background = np.asarray(background, dtype=float)
        if background.ndim != 2 or background.shape[1] != len(feature_names):
            raise ValueError("background must be (k, n_features)")
        self.predict_fn = predict_fn
        self.background = background
        self.names = list(feature_names)
        self.d = len(self.names)
        self.chunk_rows = chunk_rows
        if self.d > 12:
            raise ValueError("Exact enumeration is only meant for <= 12 features")

        # all subsets as a boolean matrix (2^d, d); row index == bitmask
        masks = ((np.arange(2**self.d)[:, None] >> np.arange(self.d)) & 1).astype(bool)
        self._masks = masks
        self._sizes = masks.sum(axis=1)

    def _predict_chunked(self, X: np.ndarray) -> np.ndarray:
        out = []
        for i in range(0, len(X), self.chunk_rows):
            out.append(np.asarray(self.predict_fn(X[i : i + self.chunk_rows]), dtype=float))
        return np.concatenate(out)

    def explain(self, x: Sequence[float]) -> Dict:
        """Return exact Shapley values (in probability units) for one patient."""
        x = np.asarray(x, dtype=float)
        if x.shape != (self.d,):
            raise ValueError(f"x must have {self.d} values")
        k = len(self.background)
        n_sub = len(self._masks)

        # Build (n_sub * k, d): for each subset take x where mask is True, else background row.
        X = np.where(
            self._masks[:, None, :],  # (n_sub, 1, d)
            x[None, None, :],
            self.background[None, :, :],  # (1, k, d)
        ).reshape(n_sub * k, self.d)

        preds = self._predict_chunked(X).reshape(n_sub, k)
        v = preds.mean(axis=1)  # value of every subset

        base_value = float(v[0])  # empty subset = average background prediction
        full_value = float(v[-1])  # all features = the patient's own prediction

        d = self.d
        phi = np.zeros(d)
        for i in range(d):
            bit = 1 << i
            for mask_idx in range(n_sub):
                if mask_idx & bit:
                    continue  # subset must NOT contain i
                s = int(self._sizes[mask_idx])
                w = factorial(s) * factorial(d - s - 1) / factorial(d)
                phi[i] += w * (v[mask_idx | bit] - v[mask_idx])

        return {
            "unit": "probability",
            "base_value": base_value,
            "prediction": full_value,
            "values": {n: float(p) for n, p in zip(self.names, phi)},
            "additivity_error": float(abs(base_value + phi.sum() - full_value)),
        }


def top_factors(explanation: Dict, k: int = 6) -> List[Dict]:
    """Sorted list of {feature, effect} by absolute effect. `effect` is in probability points (0-1)."""
    items = sorted(explanation["values"].items(), key=lambda kv: abs(kv[1]), reverse=True)
    return [
        {"feature": f, "effect": e, "direction": "raises" if e > 0 else "lowers"}
        for f, e in items[:k]
    ]


def choose_background(X, k: int, random_state: int = 0) -> np.ndarray:
    """
    k REAL rows that summarise X: the row nearest to each k-means centre.
    A small background (k = 4) is used because every extra row costs 256 model calls per patient;
    choosing representative rows keeps the baseline close to the population average.
    Missing values are only filled for the clustering; the returned rows keep their blanks.
    """
    from sklearn.cluster import KMeans

    A = np.asarray(X, dtype=float)
    if k >= len(A):
        return A.copy()
    B = np.where(np.isnan(A), np.nanmedian(A, axis=0), A)
    sd = B.std(axis=0); sd[sd == 0] = 1.0
    Z = (B - B.mean(axis=0)) / sd
    centres = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(Z).cluster_centers_
    idx: List[int] = []
    for c in centres:
        for j in np.argsort(((Z - c) ** 2).sum(axis=1)):
            if int(j) not in idx:
                idx.append(int(j)); break
    return A[idx]


# --------------------------------------------------------------------------------------
# Counterfactuals (grid search over Glucose and BMI)
# --------------------------------------------------------------------------------------
def find_counterfactuals(
    x: Sequence[float],
    feature_names: Sequence[str],
    predict_fn: PredictFn,
    threshold: float,
    glucose_floor: float = 70.0,
    bmi_floor: float = 18.5,
    glucose_step: float = 5.0,
    bmi_step: float = 0.5,
    max_results: int = 3,
    max_glucose_drop: float = 60.0,
    max_bmi_drop: float = 8.0,
) -> Dict:
    """
    Smallest changes in Glucose and/or BMI that bring the score below `threshold`.

    Only LOWERING is considered, and never below `glucose_floor` / `bmi_floor`
    (we do not want to suggest unhealthy low values). Features that are missing (NaN)
    are left unchanged and are not varied.

    The search window is limited to `max_glucose_drop` mg/dL and `max_bmi_drop` BMI points below the
    current values (at most about 13 x 17 = 221 model rows, one batched call). Options beyond the
    40 mg/dL / 5 BMI-point limits below are still marked `realistic: False`.
    """
    names = list(feature_names)
    x = np.asarray(x, dtype=float)
    gi, bi = names.index("Glucose"), names.index("BMI")
    g0, b0 = x[gi], x[bi]

    g_lo = max(glucose_floor, g0 - max_glucose_drop) if not np.isnan(g0) else 0.0
    b_lo = max(bmi_floor, b0 - max_bmi_drop) if not np.isnan(b0) else 0.0
    g_vals = [g0] if np.isnan(g0) else sorted({g0, *np.arange(g_lo, g0, glucose_step)} if g0 > g_lo else {g0})
    b_vals = [b0] if np.isnan(b0) else sorted({b0, *np.arange(b_lo, b0, bmi_step)} if b0 > b_lo else {b0})

    grid = np.array([[g, b] for g in g_vals for b in b_vals], dtype=float)
    X = np.tile(x, (len(grid), 1))
    X[:, gi] = grid[:, 0]
    X[:, bi] = grid[:, 1]
    p = np.asarray(predict_fn(X), dtype=float)  # ONE batched call

    current_p = float(np.asarray(predict_fn(x[None, :]), dtype=float)[0])

    # change "cost": how far each value moves, relative to a plausible range
    g_range, b_range = 100.0, 15.0
    dg = np.zeros(len(grid)) if np.isnan(g0) else g0 - grid[:, 0]
    db = np.zeros(len(grid)) if np.isnan(b0) else b0 - grid[:, 1]
    cost = dg / g_range + db / b_range
    g_unchanged = (dg == 0)
    b_unchanged = (db == 0)

    ok = np.where(p < threshold)[0]
    result = {
        "unit": "probability",
        "current_score": current_p,
        "threshold": threshold,
        "already_below_threshold": current_p < threshold,
        "options": [],
        "achievable": bool(len(ok)) and current_p >= threshold,
        "lowest_reachable_score": float(p.min()),
        "method": "grid search over Glucose and BMI (lowering only, limited window), one batched model call",
    }
    if current_p < threshold:
        return result

    if len(ok) == 0:
        best = int(np.argmin(p))
        result["lowest_reachable_option"] = _option(grid[best], p[best], g0, b0)
        return result

    # three diverse options: glucose only, BMI only, cheapest mix
    picks: Dict[str, int] = {}
    g_only = ok[b_unchanged[ok]]  # only glucose moved
    if len(g_only):
        picks["glucose_only"] = int(g_only[np.argmin(cost[g_only])])
    b_only = ok[g_unchanged[ok]]  # only BMI moved
    if len(b_only):
        picks["bmi_only"] = int(b_only[np.argmin(cost[b_only])])
    picks["smallest_combined_change"] = int(ok[np.argmin(cost[ok])])

    seen = set()
    for label, idx in picks.items():
        key = (round(grid[idx, 0], 3), round(grid[idx, 1], 3))
        if key in seen:
            continue
        seen.add(key)
        opt = _option(grid[idx], p[idx], g0, b0)
        opt["kind"] = label
        result["options"].append(opt)
        if len(result["options"]) >= max_results:
            break
    return result


# Design choice (not a clinical rule): changes bigger than this are flagged as "not realistic",
# so the narrator never presents them as an easy goal.
MAX_REALISTIC_GLUCOSE_DROP = 40.0  # mg/dL
MAX_REALISTIC_BMI_DROP = 5.0       # BMI points


def _option(row: np.ndarray, score: float, g0: float, b0: float) -> Dict:
    dg = 0.0 if np.isnan(g0) or np.isnan(row[0]) else g0 - row[0]
    db = 0.0 if np.isnan(b0) or np.isnan(row[1]) else b0 - row[1]
    return {
        "realistic": bool(dg <= MAX_REALISTIC_GLUCOSE_DROP and db <= MAX_REALISTIC_BMI_DROP),
        "Glucose": None if np.isnan(row[0]) else float(row[0]),
        "BMI": None if np.isnan(row[1]) else float(round(row[1], 1)),
        "glucose_change": None if np.isnan(g0) else float(round(row[0] - g0, 1)),
        "bmi_change": None if np.isnan(b0) else float(round(row[1] - b0, 1)),
        "new_score": float(score),
    }


In [ ]:
%%writefile benchmark.py
"""
HealthLens model benchmark: TabPFN-3.5 (primary) vs XGBoost vs Logistic Regression.

Protocol (same as the earlier project work, so numbers are comparable):
  * Pima Indians Diabetes, 768 rows, stratified 80/20 split  ->  614 train / 154 test
  * Zeros in Glucose, BloodPressure, SkinThickness, Insulin, BMI mean "not measured" -> NaN
  * Model selection and thresholds use TRAIN data only (5-fold CV, out-of-fold predictions)
  * The 154 test rows are touched once, at the end
  * Decision threshold = highest-precision threshold that still reaches recall >= 0.80
    on the train out-of-fold predictions

Run in Colab (notebook calls `run_benchmark`) or locally:
    python benchmark.py --data ../data/pima-indians-diabetes.csv --out results --no-tabpfn
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path
from typing import Callable, Dict

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# IMPORTANT: set this to the same value as RANDOM_STATE in your ml/src/train.py
# so the train/test split matches your earlier results.
RANDOM_STATE = 42
TARGET_RECALL = 0.80

FEATURES = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age",
]
ZERO_MEANS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]


# ----------------------------------------------------------------------------- data
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=FEATURES + ["Outcome"])
    for c in ZERO_MEANS_MISSING:
        df[c] = df[c].replace(0, np.nan)
    assert len(df) == 768 and df.Outcome.sum() == 268, "unexpected dataset"
    return df


def split(df: pd.DataFrame):
    X, y = df[FEATURES], df["Outcome"]
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


# ----------------------------------------------------------------------------- models
# TabPFN settings used everywhere (benchmark AND server). On a CPU, the defaults take about 45 s per call;
# caching the training context at fit time and using one ensemble member brings one patient to ~0.1 s
# (measured on a 2-CPU Colab runtime: test ROC-AUC 0.822 with and without these settings).
TABPFN_SETTINGS = {"fit_mode": "fit_with_cache", "n_estimators": 1}


def make_tabpfn():
    """TabPFN-3.5. Reads the token from the TABPFN_TOKEN environment variable (never hard-code it)."""
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion

    return TabPFNClassifier.create_default_for_version(ModelVersion.V3_5, random_state=RANDOM_STATE, **TABPFN_SETTINGS)


def make_xgboost():
    from xgboost import XGBClassifier

    base = XGBClassifier(
        eval_metric="logloss", subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=1, tree_method="hist",
    )
    grid = {"max_depth": [2, 3, 4], "n_estimators": [100, 300], "learning_rate": [0.03, 0.1]}
    return GridSearchCV(base, grid, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                        scoring="roc_auc", n_jobs=1)


def make_logreg():
    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ])
    grid = {"lr__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}
    return GridSearchCV(pipe, grid, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                        scoring="roc_auc", n_jobs=1)


def default_models(include_tabpfn: bool = True) -> Dict[str, object]:
    models = {}
    if include_tabpfn:
        models["TabPFN-3.5"] = make_tabpfn()
    models["XGBoost"] = make_xgboost()
    models["LogReg"] = make_logreg()
    return models


# ----------------------------------------------------------------------------- metrics
def choose_threshold(y_true, proba, target_recall=TARGET_RECALL) -> float:
    """Highest-precision threshold that still reaches target_recall (ties -> higher threshold)."""
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    precision, recall = precision[:-1], recall[:-1]
    ok = np.where(recall >= target_recall)[0]
    if len(ok) == 0:
        return float(thresholds[0])
    idx = ok[np.lexsort((thresholds[ok], precision[ok]))[-1]]
    return float(thresholds[idx])


def metrics_at(y_true, proba, thr) -> Dict:
    pred = (np.asarray(proba) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "threshold": round(float(thr), 4),
        "recall": round(tp / (tp + fn), 3),
        "precision": round(tp / (tp + fp), 3) if (tp + fp) else None,
        "specificity": round(tn / (tn + fp), 3),
        "accuracy": round((tp + tn) / len(pred), 3),
        "tp": int(tp), "fn": int(fn), "fp": int(fp), "tn": int(tn),
    }


def bootstrap_auc_ci(y, p, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        vals.append(roc_auc_score(y[i], p[i]))
    return [round(float(np.percentile(vals, 2.5)), 3), round(float(np.percentile(vals, 97.5)), 3)]


def time_predict(fitted, X: pd.DataFrame, repeats=3) -> Dict[str, float]:
    """Seconds to score 1 row, 625 rows (counterfactual grid) and 5120 rows (one Shapley explanation)."""
    out = {}
    for n in (1, 625, 5120):
        Xn = pd.concat([X] * (n // len(X) + 1), ignore_index=True).iloc[:n]
        ts = []
        for _ in range(repeats):
            t = time.perf_counter(); fitted.predict_proba(Xn); ts.append(time.perf_counter() - t)
        out[f"rows_{n}"] = round(float(np.median(ts)), 3)
    return out


# ----------------------------------------------------------------------------- main
def run_benchmark(data_path: str, out_dir: str, models: Dict[str, object] | None = None,
                  n_boot: int = 2000, log: Callable[[str], None] = print) -> Dict:
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = load_data(data_path)
    Xtr, Xte, ytr, yte = split(df)
    log(f"train {len(Xtr)} ({int(ytr.sum())} positive) | test {len(Xte)} ({int(yte.sum())} positive)")
    models = models or default_models()
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

    results, test_proba, fitted_models = {}, {}, {}
    for name, est in models.items():
        log(f"\n=== {name} ===")
        t0 = time.perf_counter()
        # honest CV score of the whole procedure (including any tuning) on TRAIN only
        cv_auc = cross_val_score(clone(est), Xtr, ytr, cv=cv, scoring="roc_auc")
        oof = cross_val_predict(clone(est), Xtr, ytr, cv=cv, method="predict_proba")[:, 1]
        thr = choose_threshold(ytr, oof)
        log(f"CV ROC-AUC {cv_auc.mean():.3f} +/- {cv_auc.std():.3f} | threshold {thr:.3f} | {time.perf_counter()-t0:.0f}s")

        fit_t = time.perf_counter()
        final = clone(est).fit(Xtr, ytr)
        fit_s = time.perf_counter() - fit_t
        p = final.predict_proba(Xte)[:, 1]
        test_proba[name], fitted_models[name] = p, final

        res = {
            "cv_roc_auc_mean": round(float(cv_auc.mean()), 4),
            "cv_roc_auc_std": round(float(cv_auc.std()), 4),
            "test_roc_auc": round(float(roc_auc_score(yte, p)), 4),
            "test_roc_auc_ci95": bootstrap_auc_ci(yte, p, n_boot),
            "test_pr_auc": round(float(average_precision_score(yte, p)), 4),
            "test_brier": round(float(brier_score_loss(yte, p)), 4),
            "at_chosen_threshold": metrics_at(yte, p, thr),
            "at_0.5": metrics_at(yte, p, 0.5),
            "fit_seconds": round(fit_s, 2),
            "predict_seconds": time_predict(final, Xte),
        }
        results[name] = res
        log(json.dumps({k: res[k] for k in ("test_roc_auc", "test_roc_auc_ci95", "test_pr_auc", "test_brier")}))
        log("at threshold: " + json.dumps(res["at_chosen_threshold"]))
        log("predict seconds: " + json.dumps(res["predict_seconds"]))

    majority = round(1 - float(yte.mean()), 3)
    summary = {
        "n_train": int(len(Xtr)), "n_test": int(len(Xte)), "random_state": RANDOM_STATE,
        "target_recall": TARGET_RECALL, "majority_class_accuracy": majority, "models": results,
    }
    (out / "benchmark_results.json").write_text(json.dumps(summary, indent=2))
    _plots(out, yte, test_proba, results)
    _table(out, results)
    # train/test CSVs (the backend needs the 614 training rows; the test rows are for your own checks)
    pd.concat([Xtr, ytr], axis=1).to_csv(out / "train.csv", index=False)
    pd.concat([Xte, yte], axis=1).to_csv(out / "test.csv", index=False)
    return {"summary": summary, "fitted": fitted_models, "test_proba": test_proba, "X_train": Xtr, "y_train": ytr,
            "X_test": Xte, "y_test": yte}


def _table(out: Path, results: Dict):
    rows = []
    for n, r in results.items():
        t = r["at_chosen_threshold"]
        rows.append({"Model": n, "CV AUC": f"{r['cv_roc_auc_mean']:.3f} ± {r['cv_roc_auc_std']:.3f}",
                     "Test AUC": f"{r['test_roc_auc']:.3f}", "95% CI": f"{r['test_roc_auc_ci95'][0]:.2f}-{r['test_roc_auc_ci95'][1]:.2f}",
                     "PR-AUC": f"{r['test_pr_auc']:.3f}", "Brier": f"{r['test_brier']:.3f}",
                     "Recall": f"{t['recall']:.2f}", "Precision": f"{t['precision']:.2f}", "Specificity": f"{t['specificity']:.2f}",
                     "1-row s": r["predict_seconds"]["rows_1"], "5120-row s": r["predict_seconds"]["rows_5120"]})
    pd.DataFrame(rows).to_csv(out / "comparison_table.csv", index=False)


def _plots(out: Path, y, proba: Dict[str, np.ndarray], results: Dict):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.calibration import calibration_curve
    from sklearn.metrics import roc_curve

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for n, p in proba.items():
        fpr, tpr, _ = roc_curve(y, p)
        ax[0].plot(fpr, tpr, label=f"{n} (AUC {results[n]['test_roc_auc']:.3f})")
        frac, mean = calibration_curve(y, p, n_bins=5, strategy="quantile")
        ax[1].plot(mean, frac, marker="o", label=n)
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set(title="ROC curve (154 test patients)", xlabel="False positive rate", ylabel="True positive rate")
    ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(title="Calibration (reliability)", xlabel="Predicted probability", ylabel="Observed fraction")
    ax[0].legend(); ax[1].legend(); fig.tight_layout(); fig.savefig(out / "roc_calibration.png", dpi=150); plt.close(fig)


# ----------------------------------------------------------------------------- export for the backend
BACKGROUND_K = 4  # rows used as the SHAP background (256 x 4 = 1,024 model rows per patient); read by the backend from the model card


def global_importance(fitted, X_test: pd.DataFrame, y_test, n_repeats: int = 10) -> Dict[str, Dict[str, float]]:
    """Permutation importance on the test set: how much ROC-AUC drops when one feature is shuffled."""
    from sklearn.inspection import permutation_importance

    r = permutation_importance(fitted, X_test, y_test, scoring="roc_auc", n_repeats=n_repeats, random_state=RANDOM_STATE)
    return {f: {"auc_drop": round(float(m), 4), "std": round(float(s), 4)}
            for f, m, s in sorted(zip(X_test.columns, r.importances_mean, r.importances_std), key=lambda t: -t[1])}


def export_artifacts(res: Dict, primary: str, out_dir: str, package_version: str = "") -> Path:
    """Write artifacts/model_card.json + artifacts/train.csv that the backend loads at start-up."""
    art = Path(out_dir) / "artifacts"; art.mkdir(parents=True, exist_ok=True)
    summ = res["summary"]; m = summ["models"][primary]; t = m["at_chosen_threshold"]
    fa = round((1 - t["precision"]) * 100) if t["precision"] else None
    card = {
        "model": {"name": primary, "tabpfn_package_version": package_version,
                  "settings": TABPFN_SETTINGS if primary.startswith("TabPFN") else {}},
        "features": FEATURES,
        "zero_means_missing": ZERO_MEANS_MISSING,
        "threshold": t["threshold"],
        "target_recall": summ["target_recall"],
        "n_train": summ["n_train"], "n_test": summ["n_test"], "random_state": summ["random_state"],
        "background_k": BACKGROUND_K,
        "cv_roc_auc": {"mean": m["cv_roc_auc_mean"], "std": m["cv_roc_auc_std"]},
        "test": {k: m[k] for k in ("test_roc_auc", "test_roc_auc_ci95", "test_pr_auc", "test_brier")},
        "at_chosen_threshold": t,
        "at_0.5": m["at_0.5"],
        "majority_class_accuracy": summ["majority_class_accuracy"],
        "comparison": {n: {"cv_roc_auc": r["cv_roc_auc_mean"], "test_roc_auc": r["test_roc_auc"]}
                       for n, r in summ["models"].items()},
        "predict_seconds": m["predict_seconds"],
        "global_importance": global_importance(res["fitted"][primary], res["X_test"], res["y_test"]),
        "limitations": [
            "Trained on 768 women (Pima Indian heritage, age 21+) from one study; it may not transfer to other groups.",
            "The test set is small (154 patients), so every metric has wide uncertainty.",
            f"Tuned to catch most higher-risk patients, so about {fa}% of people flagged will not have diabetes." if fa is not None else "",
            "Glucose in this dataset is a 2-hour oral glucose tolerance test value, not a fasting or random reading.",
            "This is a screening aid, not a diagnosis. Talk to a clinician about any result.",
        ],
    }
    card["limitations"] = [x for x in card["limitations"] if x]
    (art / "model_card.json").write_text(json.dumps(card, indent=2))
    pd.concat([res["X_train"], res["y_train"]], axis=1).to_csv(art / "train.csv", index=False)
    return art


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results")
    ap.add_argument("--no-tabpfn", action="store_true")
    a = ap.parse_args()
    run_benchmark(a.data, a.out, default_models(include_tabpfn=not a.no_tabpfn))


## Data
Downloads the public Pima dataset (768 rows). If this cell fails, upload `pima-indians-diabetes.csv` to the Colab file panel and rename it `pima.csv`.

In [ ]:
!wget -q -O pima.csv https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv
!wc -l pima.csv

## Run the comparison
**Check first:** `RANDOM_STATE` in `benchmark.py` must equal `RANDOM_STATE` in your `ml/src/train.py`. If it differs, change it in the cell above (the `%%writefile benchmark.py` cell) and run again.
XGBoost and Logistic Regression are tuned with a small grid inside cross-validation. TabPFN is used with its defaults.
This takes a few minutes.

In [ ]:
import pandas as pd, json
import benchmark as b
USE_TABPFN = True
PRIMARY = "TabPFN-3.5" if USE_TABPFN else "LogReg"
res = b.run_benchmark("pima.csv", "results", b.default_models(include_tabpfn=USE_TABPFN))

In [ ]:
from IPython.display import display, Image
display(pd.read_csv("results/comparison_table.csv"))
display(Image("results/roc_calibration.png"))
print("Majority-class accuracy on the test set:", res["summary"]["majority_class_accuracy"])

## Speed check: will explanations feel fast?
**Important:** the live server runs on a CPU. In Colab use *Runtime > Change runtime type > CPU* before running this notebook (or at least this section), otherwise a GPU will make the timings look better than they will be in production.
Exact SHAP needs 1,024 predictions per patient (256 feature subsets x 4 background rows). The counterfactual search needs at most about 221.

In [ ]:
import time, numpy as np
from explain_agnostic import ShapleyExplainer, choose_background, find_counterfactuals, top_factors
import torch
print("device:", "GPU (timings will be optimistic, re-run on CPU)" if torch.cuda.is_available() else "CPU (matches the live server)")
model = res["fitted"][PRIMARY]
Xtr, Xte = res["X_train"], res["X_test"]
f = lambda X: model.predict_proba(pd.DataFrame(X, columns=b.FEATURES))[:, 1]
f(Xte.values[:1])   # warm-up call
bg = choose_background(Xtr.values, b.BACKGROUND_K)
ex = ShapleyExplainer(f, bg, b.FEATURES)

x = Xte.iloc[0].values
t = time.time(); e = ex.explain(x); shap_s = time.time() - t
print(f"Exact SHAP: {shap_s:.1f} s | additivity error {e['additivity_error']:.2e}")
print(json.dumps(top_factors(e, 4), indent=1))

thr = res["summary"]["models"][PRIMARY]["at_chosen_threshold"]["threshold"]
hi = Xte.iloc[[int(i) for i in np.argsort(-f(Xte.values))[:1]]].values[0]
t = time.time(); cf = find_counterfactuals(hi, b.FEATURES, f, thr); cf_s = time.time() - t
print(f"Counterfactual search: {cf_s:.1f} s")
print(json.dumps(cf, indent=1)[:1200])
print("\nGUIDE: under ~3 s each is comfortable for the live app. Between 3 and 15 s, show a 'working on it' spinner.")
print("Expected on a 2-CPU runtime: about 8 s for SHAP and about 2 s for the counterfactual search. Send me both numbers.")

## Export for the backend

In [ ]:
import importlib.metadata as md_
art = b.export_artifacts(res, PRIMARY, "results", package_version=md_.version("tabpfn") if USE_TABPFN else "")
print(open(art / "model_card.json").read()[:600])
!cd results && zip -qr healthlens_artifacts.zip artifacts benchmark_results.json comparison_table.csv roc_calibration.png test.csv
from google.colab import files
files.download("results/healthlens_artifacts.zip")